In [1]:
from pathlib import Path
import pandas as pd

In [2]:
DATA_DIR = Path("../data/raw/")
SALES_PATH = DATA_DIR / "sales_train_evaluation.csv"
CALENDAR_PATH = DATA_DIR / "calendar.csv"
PRICES_PATH = DATA_DIR / "sell_prices.csv"

## Load & Melt Sales Data

> **Note:**
>
> Due to the high volume of data, it can take a high amount of memory to load
> and store the data. To fix this, use category dtypes wherever possible.
> Otherwise, the merges and intense operations can cause crashes.

In [ ]:
def load_csv_with_correct_dtypes(path: Path) -> pd.DataFrame:
    category_cols = [
        "id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "d",
    ]
    df = pd.read_csv(path)
    df = df.astype({c: "category" for c in category_cols if c in df.columns})
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    return df


sales = load_csv_with_correct_dtypes(SALES_PATH)
calendar = load_csv_with_correct_dtypes(CALENDAR_PATH)
prices = load_csv_with_correct_dtypes(PRICES_PATH)

# Filter for one store for testing (will run feature engineering on full
# dataset using Azure)
sales = sales.query("store_id == 'CA_1'").reset_index(drop=True)
prices = prices.query("store_id == 'CA_1'").reset_index(drop=True)

TokenError: ('unterminated string literal (detected at line 1)', (1, 13))

In [11]:
day_cols = [c for c in sales.columns if c.startswith("d_")]
id_cols = [c for c in sales.columns if not c.startswith("d_")]

sales_long = sales.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales",
)
sales_long["d"] = sales_long["d"].astype("category")

sales_long.info()
sales_long.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5918109 entries, 0 to 5918108
Data columns (total 8 columns):
 #   Column    Dtype   
---  ------    -----   
 0   id        category
 1   item_id   category
 2   dept_id   category
 3   cat_id    category
 4   store_id  category
 5   state_id  category
 6   d         category
 7   sales     int64   
dtypes: category(7), int64(1)
memory usage: 103.0 MB


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


## Merge Calendar & Prices

In [12]:
# Merge Calendar
df = sales_long.merge(calendar, how="left", on="d")
df["d"] = df["d"].astype("category")

print(f"Date range:  {df['date'].min()}  -  {df['date'].max()}")
print()
df.info()
df.head()

Date range:  2011-01-29 00:00:00  -  2016-05-22 00:00:00

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5918109 entries, 0 to 5918108
Data columns (total 21 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            category      
 1   item_id       category      
 2   dept_id       category      
 3   cat_id        category      
 4   store_id      category      
 5   state_id      category      
 6   d             category      
 7   sales         int64         
 8   date          datetime64[ns]
 9   wm_yr_wk      int64         
 10  weekday       object        
 11  wday          int64         
 12  month         int64         
 13  year          int64         
 14  event_name_1  object        
 15  event_type_1  object        
 16  event_name_2  object        
 17  event_type_2  object        
 18  snap_CA       int64         
 19  snap_TX       int64         
 20  snap_WI       int64         
dtypes: category(7), datetime64[ns](1), int64(8

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0


In [13]:
# Merge Prices
df = df.merge(prices, how="left", on=["store_id", "item_id", "wm_yr_wk"])
df = df.sort_values(["store_id", "item_id", "date"], ignore_index=True)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5918109 entries, 0 to 5918108
Data columns (total 22 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            category      
 1   item_id       category      
 2   dept_id       category      
 3   cat_id        category      
 4   store_id      category      
 5   state_id      category      
 6   d             category      
 7   sales         int64         
 8   date          datetime64[ns]
 9   wm_yr_wk      int64         
 10  weekday       object        
 11  wday          int64         
 12  month         int64         
 13  year          int64         
 14  event_name_1  object        
 15  event_type_1  object        
 16  event_name_2  object        
 17  event_type_2  object        
 18  snap_CA       int64         
 19  snap_TX       int64         
 20  snap_WI       int64         
 21  sell_price    float64       
dtypes: category(7), datetime64[ns](1), float64(1), int64(8), object(5)

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,2,2011,NaN,NaN,NaN,NaN,1,1,0,2.0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,2,2011,NaN,NaN,NaN,NaN,1,0,1,2.0


In [ ]:
# Impute sell price
df["sell_price"] = (
    df
    .groupby(["store_id", "item_id"], observed=True)
    ["sell_price"].transform(lambda s: s.ffill().bfill())
)
item_mean = df.groupby("item_id")["sell_price"].transform("mean")
df["sell_price"] = df["sell_price"].fillna(item_mean)

print("Null prices:", df["sell_price"].isna().sum())
print()
df.info()
df.head()

Null prices: 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5918109 entries, 0 to 5918108
Data columns (total 22 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            category      
 1   item_id       category      
 2   dept_id       category      
 3   cat_id        category      
 4   store_id      category      
 5   state_id      category      
 6   d             category      
 7   sales         int64         
 8   date          datetime64[ns]
 9   wm_yr_wk      int64         
 10  weekday       object        
 11  wday          int64         
 12  month         int64         
 13  year          int64         
 14  event_name_1  object        
 15  event_type_1  object        
 16  event_name_2  object        
 17  event_type_2  object        
 18  snap_CA       int64         
 19  snap_TX       int64         
 20  snap_WI       int64         
 21  sell_price    float64       
dtypes: category(7), datetime64[ns](1), float64(1), int

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,2,2011,NaN,NaN,NaN,NaN,1,1,0,2.0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,2,2011,NaN,NaN,NaN,NaN,1,0,1,2.0


## Lag Features

In [ ]:
df = df.sort_values(["store_id", "item_id", "date"], ignore_index=True)

for lag in [7, 14, 28]:
    df[f"sales_lag_{lag}"] = (
        df
        .groupby(["store_id", "item_id", "sales"])
        ["sales"].shift(lag)
    )

df.info()
df.tail()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5918109 entries, 0 to 5918108
Data columns (total 25 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            category      
 1   item_id       category      
 2   dept_id       category      
 3   cat_id        category      
 4   store_id      category      
 5   state_id      category      
 6   d             category      
 7   sales         int64         
 8   date          datetime64[ns]
 9   wm_yr_wk      int64         
 10  weekday       object        
 11  wday          int64         
 12  month         int64         
 13  year          int64         
 14  event_name_1  object        
 15  event_type_1  object        
 16  event_name_2  object        
 17  event_type_2  object        
 18  snap_CA       int64         
 19  snap_TX       int64         
 20  snap_WI       int64         
 21  sell_price    float64       
 22  sales_lag_7   float64       
 23  sales_lag_14  float64       
 24

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,sales_lag_7,sales_lag_14,sales_lag_28
5918104,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1937,0,2016-05-18,11616,...,NaN,NaN,NaN,0,0,0,5.94,0.0,0.0,0.0
5918105,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1938,0,2016-05-19,11616,...,NaN,NaN,NaN,0,0,0,5.94,0.0,0.0,0.0
5918106,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1939,1,2016-05-20,11616,...,NaN,NaN,NaN,0,0,0,5.94,1.0,1.0,1.0
5918107,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1940,0,2016-05-21,11617,...,NaN,NaN,NaN,0,0,0,5.94,0.0,0.0,0.0
5918108,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1941,0,2016-05-22,11617,...,NaN,NaN,NaN,0,0,0,5.94,0.0,0.0,0.0


## Rolling Statistics

In [20]:
for window in [7, 28]:
    shifted = df.groupby(["store_id", "item_id"])["sales"].shift(1)

    df[f"rolling_mean_{window}"] = (
        shifted
        .groupby(df["store_id"].astype(str) + "_" + df["item_id"].astype(str))
        .transform(lambda s: s.rolling(window, min_periods=1).mean())
    )
    df[f"rolling_std_{window}"] = (
        shifted
        .groupby(df["store_id"].astype(str) + "_" + df["item_id"].astype(str))
        .transform(lambda s: s.rolling(window, min_periods=1).std())
    )

df.tail()

/tmp/ipykernel_3086/1274767509.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  shifted = df.groupby(["store_id", "item_id"])["sales"].shift(1)
/tmp/ipykernel_3086/1274767509.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  shifted = df.groupby(["store_id", "item_id"])["sales"].shift(1)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,snap_TX,snap_WI,sell_price,sales_lag_7,sales_lag_14,sales_lag_28,rolling_mean_7,rolling_std_7,rolling_mean_28,rolling_std_28
5918104,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1937,0,2016-05-18,11616,...,0,0,5.94,0.0,0.0,0.0,0.142857,0.377964,0.142857,0.356348
5918105,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1938,0,2016-05-19,11616,...,0,0,5.94,0.0,0.0,0.0,0.000000,0.000000,0.142857,0.356348
5918106,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1939,1,2016-05-20,11616,...,0,0,5.94,1.0,1.0,1.0,0.000000,0.000000,0.107143,0.314970
5918107,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1940,0,2016-05-21,11617,...,0,0,5.94,0.0,0.0,0.0,0.142857,0.377964,0.107143,0.314970
5918108,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1941,0,2016-05-22,11617,...,0,0,5.94,0.0,0.0,0.0,0.142857,0.377964,0.107143,0.314970


## Calendar Features

In [21]:
df["day_of_week"] = df["date"].dt.dayofweek   # 0 = Monday
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_holiday"] = df["event_type_1"].notna().astype(int)
df["is_snap_CA"] = df["snap_CA"].astype(int)
df["is_snap_TX"] = df["snap_TX"].astype(int)
df["is_snap_WI"] = df["snap_WI"].astype(int)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5918109 entries, 0 to 5918108
Data columns (total 36 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               category      
 1   item_id          category      
 2   dept_id          category      
 3   cat_id           category      
 4   store_id         category      
 5   state_id         category      
 6   d                category      
 7   sales            int64         
 8   date             datetime64[ns]
 9   wm_yr_wk         int64         
 10  weekday          object        
 11  wday             int64         
 12  month            int32         
 13  year             int64         
 14  event_name_1     object        
 15  event_type_1     object        
 16  event_name_2     object        
 17  event_type_2     object        
 18  snap_CA          int64         
 19  snap_TX          int64         
 20  snap_WI          int64         
 21  sell_price       float64       

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,rolling_std_7,rolling_mean_28,rolling_std_28,day_of_week,week_of_year,is_weekend,is_holiday,is_snap_CA,is_snap_TX,is_snap_WI
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,NaN,NaN,NaN,5,4,1,0,0,0,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,NaN,3.0,NaN,6,4,1,0,0,0,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,2.121320,1.5,2.121320,0,5,0,0,0,0,0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,1.732051,1.0,1.732051,1,5,0,0,1,1,0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,1.414214,1.0,1.414214,2,5,0,0,1,0,1


## Price Features

In [22]:
# Week-over-week price change within item-store
df["price_change_pct"] = (
    df
    .groupby(["store_id", "item_id"])
    ["sell_price"].pct_change()
    .fillna(0)
)

df["is_price_decrease"] = (df["price_change_pct"] < 0).astype(int)
df["is_price_increase"] = (df["price_change_pct"] > 0).astype(int)

# Relative price vs. category mean across stores
# (computed as a feature, not imputation)
category_mean = df.groupby(["dept_id", "date"])["sell_price"].transform("mean")
df["price_relative_to_category_mean"] = df["sell_price"] / category_mean

df.info()
df.head()

/tmp/ipykernel_3086/4257644014.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["store_id", "item_id"])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5918109 entries, 0 to 5918108
Data columns (total 40 columns):
 #   Column                           Dtype         
---  ------                           -----         
 0   id                               category      
 1   item_id                          category      
 2   dept_id                          category      
 3   cat_id                           category      
 4   store_id                         category      
 5   state_id                         category      
 6   d                                category      
 7   sales                            int64         
 8   date                             datetime64[ns]
 9   wm_yr_wk                         int64         
 10  weekday                          object        
 11  wday                             int64         
 12  month                            int32         
 13  year                             int64         
 14  event_name_1                     o

/tmp/ipykernel_3086/4257644014.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  category_mean = df.groupby(["dept_id", "date"])["sell_price"].transform("mean")


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,week_of_year,is_weekend,is_holiday,is_snap_CA,is_snap_TX,is_snap_WI,price_change_pct,is_price_decrease,is_price_increase,price_relative_to_category_mean
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,4,1,0,0,0,0,0.0,0,0,0.645075
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,4,1,0,0,0,0,0.0,0,0,0.645075
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,5,0,0,0,0,0,0.0,0,0,0.645075
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,5,0,0,1,1,0,0.0,0,0,0.645075
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,5,0,0,1,0,1,0.0,0,0,0.645075


In [25]:
df["price_change_pct"].describe().round(3)

count    5918109.000
mean           0.000
std            0.104
min           -0.996
25%            0.000
50%            0.000
75%            0.000
max          246.000
Name: price_change_pct, dtype: float64